# Clustrix Configuration Manager Example

This notebook demonstrates how to use the `%%remote` magic command to manage cluster configurations interactively.

> **What this notebook actually does.** `%%remote` (the modern name for the
> old `%%clusterfy` magic, kept as a deprecated alias) displays an
> `ipywidgets` form; its "Apply Config" button calls `clustrix.configure(**config)`
> with whatever the form collected -- there is no other magic involved. Only
> `cluster_type="local"`, `"slurm"`, `"ssh"` and `"huggingface"` have been
> demonstrated running a real job end to end. The widget also lets you pick
> `"aws"`, `"gcp"`, `"azure"` and `"lambda"` cloud-provider fields (see below)
> -- those configure clustrix's cloud-VM auto-provisioning path, which is
> unverified end to end. See :ref:`limitations`.

In [ ]:
# Import clustrix -- this registers the %%remote and %%clusterfy magics,
# but does NOT display the widget. Importing a library should not inject
# UI as a side effect; run the %%remote cell below (or set the
# CLUSTRIX_AUTO_WIDGET=1 environment variable before import) to see it.
import clustrix


## Using the Configuration Widget

The `%%remote` magic command creates an interactive widget for managing cluster configurations:

In [ ]:
%%remote
# The widget interface will appear above this cell
# You can interact with it to:
# - Create new configurations
# - Edit existing configurations  
# - Apply configurations to your session
# - Save/load configurations to/from files

# Widget Screenshots and Examples:
# 
# When you run this cell, the widget will display with the default "Local Single-core" configuration:
# ![Default Widget View](../_static/img/screenshots/widget_default.png)
#
# The dropdown menu shows all available configuration templates:
# ![Configuration Dropdown](../_static/img/screenshots/widget_dropdown.png)
#
# Example SLURM cluster configuration with basic settings:
# ![SLURM Basic Configuration](../_static/img/screenshots/widget_slurm_basic.png)
#
# Advanced settings reveal additional options:
# ![SLURM Advanced Configuration](../_static/img/screenshots/widget_slurm_advanced.png)

## Widget Features

### 1. **Configuration Selection**
- Use the dropdown to select between different configurations
- Default configurations include local, AWS, GCP, Azure, SLURM, and Kubernetes options

### 2. **Configuration Management**
- **New Config**: Create a new configuration
- **Delete Config**: Remove the selected configuration
- **Apply Config**: Apply the selected configuration to your current session

### 3. **Configuration Fields**
- **Name**: Friendly name for the configuration
- **Description**: Detailed description of the cluster
- **Cluster Type**: local, ssh, slurm, pbs, sge, kubernetes, or huggingface
- **Host**: Hostname or IP address (for remote clusters)
- **Username**: SSH username (for remote clusters)
- **SSH Key**: Path to SSH private key
- **Work Dir**: Remote working directory
- **Default Cores**: Default number of CPU cores
- **Default Memory**: Default memory allocation
- **Default Time**: Default time limit

### 4. **Save/Load Configurations**
- Save configurations to YAML or JSON files
- Load configurations from files
- Share configurations with team members

## Cloud Provider Examples

The widget includes comprehensive *form* support for cloud providers -- dynamic
field visibility and intelligent defaults for the `ClusterConfig` fields each
provider uses. That is a UI/config-collection claim, not a claim that jobs run
successfully on these providers: `@cluster(provider="aws"/"gcp"/"azure"/"lambda")`
cloud-VM auto-provisioning is unverified end to end (of the built-in providers,
only `"lambda"` even implements instance creation; the others raise
`NotImplementedError` at submit time). See :ref:`limitations`.

### Google Cloud Platform
When configuring GCP, only relevant fields are displayed:

![GCP Configuration](../_static/img/screenshots/widget_gcp.png)

### Lambda Cloud GPU Instances
The widget provides specialized support for GPU-optimized Lambda Cloud instances:

![Lambda Cloud Configuration](../_static/img/screenshots/widget_lambda.png)

### Key Cloud Features
- **Dynamic Field Visibility**: Only shows fields relevant to the selected provider
- **Auto-populated Dropdowns**: Instance types, regions, and zones populated automatically
- **Provider-specific Options**: Each cloud provider has tailored configuration options
- **Cost Monitoring**: Built-in cost tracking for all cloud providers

## Example: Using a Configuration

After applying a configuration using the widget, you can use it with the `@cluster` decorator:

In [3]:
from clustrix import cluster
import numpy as np

@cluster(cores=4, memory="8GB")
def matrix_computation(size=1000):
    """Example computation that will run on the configured cluster."""
    A = np.random.rand(size, size)
    B = np.random.rand(size, size)
    C = np.dot(A, B)
    return np.mean(C)

# This will run on whatever cluster configuration is currently active
# result = matrix_computation(2000)

**Behind the scenes:** `%%remote`'s "Apply Config" button just calls
`clustrix.configure(**config)` -- it does not itself contact a cluster or
validate credentials. The configuration takes effect on the *next* call to a
`@cluster`-decorated function, not immediately: `@cluster` reads
`get_config()` fresh every time the wrapped function is called, so decoration
order relative to `configure()`/`%%remote` doesn't matter. What that call
actually does -- resource resolution, local-vs-remote choice, serialization,
submission, polling, HMAC-verified result download -- is the same order of
operations for every backend and is documented in full, source-verified
detail in :ref:`execution-model`. :ref:`configuration` documents every field
this widget can set and how `ClusterConfig` and `@cluster`'s own keyword
arguments interact.

## Programmatic Configuration

You can also check and modify configurations programmatically:

In [4]:
# Check current configuration
current_config = clustrix.get_config()
print(f"Current cluster type: {current_config.cluster_type}")
print(f"Default cores: {current_config.default_cores}")
print(f"Default memory: {current_config.default_memory}")

## Tips

1. **Save your configurations** to a file for easy sharing and version control
2. **Use descriptive names** for your configurations to easily identify them
3. **Test configurations** with small jobs before running large computations
4. **Keep SSH keys secure** and use appropriate file permissions
5. **Document cluster-specific requirements** in the description field